# S3M kNN vs GMM pipeline — map structure and ROC comparison

Compares the **S3M kNN** (original) pipeline against the **hybrid GMM** (current) pipeline
for the same epoch and sky direction (2026-05-01, antisun at ~220° ecliptic lon).

**Figures produced:**
1. Log-density comparison ±0.8 deg/day cut (S3M left | GMM right)
2. P(NEO) probability comparison ±0.8 deg/day cut
3. GMM full ±2.0 deg/day range (density + probability), showing extended coverage
4. ROC curves: S3M kNN vs GMM vs digest2 on full Sorcha simulation

**Files needed on Arnor (relative to this notebook at `neomod/`):**
```
../prob_maps/prob_maps_2026-05-01_antisun.npz       ← S3M monthly map
../prob_maps_gmm/prob_maps_2026-05-01_antisun.npz   ← GMM monthly map  (scp from Hyak)
../outputs/phase2/sorcha_comparison.parquet    ← S3M Sorcha result (scp from Hyak)
../outputs/phase2_gmm/sorcha_comparison_gmm.parquet  ← GMM result (already on Arnor)
```
Run `git pull` in `neomod/` before starting.

In [ ]:
import sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from sklearn.metrics import precision_recall_curve

sys.path.insert(0, "../../src")
import velocity_density_pipeline as vdp

mpl.rcParams.update({
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "figure.dpi": 150,
})

In [ ]:
# ── Paths — edit if your Arnor layout differs ────────────────────────────────
S3M_MAP_PATH = "../prob_maps/prob_maps_2026-05-01_antisun.npz"
GMM_MAP_PATH = "../prob_maps_gmm/prob_maps_2026-05-01_antisun.npz"
S3M_PARQUET  = "../outputs/phase2/sorcha_comparison.parquet"
GMM_PARQUET  = "../outputs/phase2_gmm/sorcha_comparison_gmm.parquet"

# Mag bins to display in comparison plots (must exist in both maps)
SHOW_BINS = ["18_20", "mag22", "mag24+"]
BIN_LABELS = {"18_20": "18–20", "mag22": "22–23", "mag24+": "24+"}

In [ ]:
# ── Load maps ─────────────────────────────────────────────────────────────────
# S3M map: nearest-dist mask ON (0.2 deg/day) — as originally used
s3m = vdp.ProbMapSet.from_npz(S3M_MAP_PATH)

# GMM map: mask OFF (inf) — as used in our Sorcha scoring
gmm = vdp.ProbMapSet.from_npz(GMM_MAP_PATH, mask_radius_deg_per_day=float('inf'))

print(f"S3M  center=({s3m.center_lon_deg:.1f}, {s3m.center_lat_deg:.1f})  "
      f"grid {s3m.x_grid[0]:.1f}..{s3m.x_grid[-1]:.1f}  ({len(s3m.x_grid)} pts)")
print(f"GMM  center=({gmm.center_lon_deg:.1f}, {gmm.center_lat_deg:.1f})  "
      f"grid {gmm.x_grid[0]:.1f}..{gmm.x_grid[-1]:.1f}  ({len(gmm.x_grid)} pts)")
print(f"Bins: {[mb['label'] for mb in s3m.mag_bins]}")
print(f"Pops: {s3m.population_names}")

In [ ]:
# ── Shared plotting helpers ───────────────────────────────────────────────────

def _log_density(density_2d):
    """Log10 density with 5-decade dynamic range. Returns (img, vmin, vmax)."""
    d = np.asarray(density_2d, float)
    pos = np.isfinite(d) & (d > 0)
    out = np.full_like(d, np.nan)
    if pos.any():
        out[pos] = np.log10(d[pos])
        vmax = float(np.nanmax(out))
        vmin = vmax - 5.0
    else:
        vmin, vmax = 0.0, 1.0
    return out, vmin, vmax


_DCMAP = mpl.colormaps["viridis"].copy()
_DCMAP.set_bad("0.88")


def _show_map(ax, data_2d, xg, yg, vmin, vmax, cmap, xlim, ylim, mode="density"):
    """imshow a (n_vlam, n_vbeta) map on ax with given axis limits."""
    if mode == "density":
        img = np.ma.masked_invalid(data_2d)
    else:
        img = np.asarray(data_2d, float)
    extent = [xg[0], xg[-1], yg[0], yg[-1]]
    ax.imshow(img.T, origin="lower", extent=extent, aspect="equal",
              vmin=vmin, vmax=vmax, cmap=cmap, interpolation="nearest")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.axhline(0, lw=0.5, color="white", ls="--", alpha=0.6)
    ax.axvline(0, lw=0.5, color="white", ls="--", alpha=0.6)


def _label_ax(ax, xlabel=True, ylabel=True):
    if xlabel:
        ax.set_xlabel(r"$v_\lambda$ (deg/day)")
    if ylabel:
        ax.set_ylabel(r"$v_\beta$ (deg/day)")


def _add_colorbar(fig, axes_row, im_or_vrange, label, cmap):
    """Attach a shared colorbar to the right of a row of axes."""
    norm = mpl.colors.Normalize(*im_or_vrange)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=axes_row, shrink=0.85, pad=0.02)
    cb.set_label(label, fontsize=9)
    return cb

## Figure 1 — Log density: S3M kNN vs GMM (±0.8 deg/day cut)

6 rows (MBA ×3 bins, NEO ×3 bins) × 2 columns (S3M | GMM).  
Shared color scale per row so you can directly compare density levels.
Both panels clipped to ±0.8 for a fair comparison on the same square.

In [ ]:
VLIM_08 = (-0.8, 0.8)

pops_rows = [("MBA", b) for b in SHOW_BINS] + [("NEO", b) for b in SHOW_BINS]
n_rows = len(pops_rows)

fig1, axes1 = plt.subplots(n_rows, 2, figsize=(8, 2.8 * n_rows),
                            constrained_layout=True)
fig1.suptitle(
    f"Log density: S3M kNN (left)  vs  GMM (right)\n"
    f"Antisun 2026-05-01, clipped to ±0.8 deg/day",
    fontsize=12, y=1.01,
)

for row_i, (pop, bin_lbl) in enumerate(pops_rows):
    ax_s, ax_g = axes1[row_i, 0], axes1[row_i, 1]

    # Pull density arrays
    d_s3m = s3m.results[bin_lbl]["density_maps_downweighted_raw"][pop]
    d_gmm = gmm.results[bin_lbl]["density_maps_downweighted_raw"][pop]

    log_s3m, vs_min, vs_max = _log_density(d_s3m)
    log_gmm, vg_min, vg_max = _log_density(d_gmm)

    # Shared scale across both panels in this row
    vmax = max(vs_max, vg_max)
    vmin = vmax - 5.0

    for ax, log_d, pms, label in [
        (ax_s, log_s3m, s3m, "S3M kNN"),
        (ax_g, log_gmm, gmm, "GMM"),
    ]:
        _show_map(ax, log_d, pms.x_grid, pms.y_grid, vmin, vmax,
                  _DCMAP, VLIM_08, VLIM_08, mode="density")
        _label_ax(ax, xlabel=(row_i == n_rows - 1), ylabel=(ax is ax_s))
        ax.set_title(f"{pop} {BIN_LABELS[bin_lbl]} mag  [{label}]", fontsize=9)

    # Shared colorbar for this row
    _add_colorbar(fig1, [ax_s, ax_g], (vmin, vmax),
                  r"$\log_{10}\langle n_0 \rangle$", _DCMAP)

plt.savefig("comparison_density_0p8cut.pdf", bbox_inches="tight", dpi=200)
plt.savefig("comparison_density_0p8cut.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved: comparison_density_0p8cut.pdf/png")

## Figure 2 — P(NEO) probability: S3M kNN vs GMM (±0.8 cut)

3 rows (NEO ×3 mag bins) × 2 columns.  
S3M loaded with mask ON (cells with nearest-clone-dist > 0.2 zeroed);  
GMM loaded with mask OFF (smooth density everywhere).

In [ ]:
fig2, axes2 = plt.subplots(len(SHOW_BINS), 2, figsize=(8, 2.8 * len(SHOW_BINS)),
                            constrained_layout=True)
fig2.suptitle(
    f"P(NEO) probability: S3M kNN (left, mask ON)  vs  GMM (right, mask OFF)\n"
    f"Antisun 2026-05-01, clipped to ±0.8 deg/day",
    fontsize=12, y=1.01,
)

for row_i, bin_lbl in enumerate(SHOW_BINS):
    ax_s, ax_g = axes2[row_i, 0], axes2[row_i, 1]

    p_s3m = s3m.get_probability_map(bin_lbl, "NEO")
    p_gmm = gmm.get_probability_map(bin_lbl, "NEO")

    for ax, prob, pms, label in [
        (ax_s, p_s3m, s3m, "S3M kNN (mask ON)"),
        (ax_g, p_gmm, gmm, "GMM (mask OFF)"),
    ]:
        _show_map(ax, prob, pms.x_grid, pms.y_grid, 0.0, 1.0,
                  "viridis", VLIM_08, VLIM_08, mode="probability")
        _label_ax(ax, xlabel=(row_i == len(SHOW_BINS) - 1), ylabel=(ax is ax_s))
        ax.set_title(f"NEO {BIN_LABELS[bin_lbl]} mag  [{label}]", fontsize=9)

    _add_colorbar(fig2, [ax_s, ax_g], (0.0, 1.0), "P(NEO)", "viridis")

plt.savefig("comparison_probability_0p8cut.pdf", bbox_inches="tight", dpi=200)
plt.savefig("comparison_probability_0p8cut.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved: comparison_probability_0p8cut.pdf/png")

## Figure 3 — GMM extended range ±2.0 deg/day

2 rows (density | probability) × 3 mag bins.  
The dashed white box marks the ±0.8 region shown in Figures 1–2.
The region from ±0.8 to ±2.0 is new coverage that rescues fast NEOs the S3M pipeline missed.

In [ ]:
VLIM_20 = (-2.0, 2.0)

fig3, axes3 = plt.subplots(2, len(SHOW_BINS), figsize=(4.5 * len(SHOW_BINS), 8),
                            constrained_layout=True)
fig3.suptitle(
    "GMM full ±2.0 deg/day range — density (top) and P(NEO) probability (bottom)\n"
    "Antisun 2026-05-01  |  dashed box = ±0.8 cut shown in Figs 1–2",
    fontsize=12, y=1.01,
)

for col_i, bin_lbl in enumerate(SHOW_BINS):
    # Row 0: log density
    ax_d = axes3[0, col_i]
    d_gmm = gmm.results[bin_lbl]["density_maps_downweighted_raw"]["NEO"]
    log_d, vmin, vmax = _log_density(d_gmm)
    _show_map(ax_d, log_d, gmm.x_grid, gmm.y_grid, vmin, vmax,
              _DCMAP, VLIM_20, VLIM_20, mode="density")
    ax_d.set_title(f"NEO density {BIN_LABELS[bin_lbl]} mag", fontsize=10)
    _label_ax(ax_d, xlabel=False, ylabel=(col_i == 0))
    _add_colorbar(fig3, [ax_d], (vmin, vmax), r"$\log_{10}\langle n_0 \rangle$", _DCMAP)

    # Row 1: probability
    ax_p = axes3[1, col_i]
    p_gmm = gmm.get_probability_map(bin_lbl, "NEO")
    _show_map(ax_p, p_gmm, gmm.x_grid, gmm.y_grid, 0.0, 1.0,
              "viridis", VLIM_20, VLIM_20, mode="probability")
    ax_p.set_title(f"P(NEO) {BIN_LABELS[bin_lbl]} mag", fontsize=10)
    _label_ax(ax_p, xlabel=True, ylabel=(col_i == 0))
    _add_colorbar(fig3, [ax_p], (0.0, 1.0), "P(NEO)", "viridis")

    # Dashed box marking the ±0.8 region
    for ax in [ax_d, ax_p]:
        rect = plt.Rectangle((-0.8, -0.8), 1.6, 1.6,
                              linewidth=1.2, edgecolor="white",
                              linestyle="--", facecolor="none", zorder=5)
        ax.add_patch(rect)

plt.savefig("gmm_extended_range_2p0.pdf", bbox_inches="tight", dpi=200)
plt.savefig("gmm_extended_range_2p0.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved: gmm_extended_range_2p0.pdf/png")

## Figure 4 — ROC comparison: S3M kNN vs GMM vs digest2

Uses the full Sorcha simulation (2yr, 597K tracklets, N_NEO=109k in-footprint).  
NaN scores (outside footprint) dropped so both classifiers are evaluated on the same subset.

In [ ]:
# Load parquets — only the columns needed for ROC
ROC_COLS = ["population", "P_NEO_vdp", "P_NEO_d2"]

print("Loading S3M parquet...")
s3m_df = pd.read_parquet(S3M_PARQUET, columns=ROC_COLS)

print("Loading GMM parquet...")
gmm_df = pd.read_parquet(GMM_PARQUET, columns=ROC_COLS)

print(f"S3M rows: {len(s3m_df):,}  |  GMM rows: {len(gmm_df):,}")
print(f"S3M NEOs: {(s3m_df.population=='NEO').sum():,}  |  "
      f"GMM NEOs: {(gmm_df.population=='NEO').sum():,}")

In [ ]:
def roc_curve_data(df):
    """Drop NaN VDP scores (outside footprint), then compute precision-recall curves."""
    sub = df.dropna(subset=["P_NEO_vdp"]).copy()
    y = (sub["population"] == "NEO").astype(int).values

    def _best_f1(scores):
        scores = np.nan_to_num(scores, nan=0.0)
        prec, rec, thr = precision_recall_curve(y, scores)
        f1 = 2 * prec * rec / (prec + rec + 1e-9)
        idx = np.argmax(f1)
        return rec, 1 - prec, f1[idx], rec[idx], 1 - prec[idx], thr[idx] if idx < len(thr) else 0.0

    vdp_rec, vdp_fpr, vdp_f1, vdp_comp, vdp_cont, vdp_thr = _best_f1(sub["P_NEO_vdp"].values)
    d2_rec,  d2_fpr,  d2_f1,  d2_comp,  d2_cont,  d2_thr  = _best_f1(sub["P_NEO_d2"].values)

    n_neo = y.sum()
    return {
        "n_neo": n_neo, "n_total": len(y),
        "vdp": {"rec": vdp_rec, "fpr": vdp_fpr, "f1": vdp_f1,
                "comp": vdp_comp, "cont": vdp_cont, "thr": vdp_thr},
        "d2":  {"rec": d2_rec,  "fpr": d2_fpr,  "f1": d2_f1,
                "comp": d2_comp,  "cont": d2_cont,  "thr": d2_thr},
    }

print("Computing ROC for S3M...")
s3m_roc = roc_curve_data(s3m_df)
print(f"  S3M VDP  F1={s3m_roc['vdp']['f1']:.3f}  "
      f"comp={s3m_roc['vdp']['comp']*100:.1f}%  cont={s3m_roc['vdp']['cont']*100:.1f}%")
print(f"  S3M d2   F1={s3m_roc['d2']['f1']:.3f}   "
      f"comp={s3m_roc['d2']['comp']*100:.1f}%  cont={s3m_roc['d2']['cont']*100:.1f}%")

print("Computing ROC for GMM...")
gmm_roc = roc_curve_data(gmm_df)
print(f"  GMM VDP  F1={gmm_roc['vdp']['f1']:.3f}  "
      f"comp={gmm_roc['vdp']['comp']*100:.1f}%  cont={gmm_roc['vdp']['cont']*100:.1f}%")
print(f"  GMM d2   F1={gmm_roc['d2']['f1']:.3f}   "
      f"comp={gmm_roc['d2']['comp']*100:.1f}%  cont={gmm_roc['d2']['cont']*100:.1f}%")

In [ ]:
fig4, ax = plt.subplots(figsize=(7, 6))

# S3M VDP
sv, sd = s3m_roc["vdp"], s3m_roc["d2"]
gv = gmm_roc["vdp"]
n_neo = gmm_roc["n_neo"]

ax.plot(sv["rec"] * 100, sv["fpr"] * 100,
        lw=1.8, color="steelblue", ls="--", label="VDP S3M kNN")
ax.plot(gv["rec"] * 100, gv["fpr"] * 100,
        lw=2.0, color="steelblue", label="VDP GMM (current)")
ax.plot(sd["rec"] * 100, sd["fpr"] * 100,
        lw=2.0, color="darkorange", ls="--", label="digest2")

# Best-F1 markers
s3m_bf = s3m_roc["vdp"]
gmm_bf = gmm_roc["vdp"]
d2_bf  = gmm_roc["d2"]

ax.scatter(s3m_bf["comp"] * 100, s3m_bf["cont"] * 100,
           s=80, color="steelblue", marker="^", zorder=5,
           label=f"S3M best F1={s3m_bf['f1']:.3f}")
ax.scatter(gmm_bf["comp"] * 100, gmm_bf["cont"] * 100,
           s=80, color="steelblue", marker="o", zorder=5,
           label=f"GMM best F1={gmm_bf['f1']:.3f}")
ax.scatter(d2_bf["comp"] * 100, d2_bf["cont"] * 100,
           s=80, color="darkorange", marker="s", zorder=5,
           label=f"digest2 best F1={d2_bf['f1']:.3f}")

ax.set_xlabel("NEO completeness: true NEOs selected (%)")
ax.set_ylabel("Contamination: non-NEOs among selected (%)")
ax.set_title(
    f"ROC: VDP S3M kNN  vs  VDP GMM  vs  digest2\n"
    f"Sorcha simulation, N_NEO={n_neo:,} (in-footprint)"
)
ax.legend(fontsize=9, loc="upper left")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig("roc_s3m_vs_gmm.pdf", bbox_inches="tight", dpi=200)
plt.savefig("roc_s3m_vs_gmm.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved: roc_s3m_vs_gmm.pdf/png")